# SignBridge AI — Step 5: Build Dataset Index

Generate `dataset_index.json` mapping each UID to pose path, video path, and text.

In [ ]:
import os
import csv
import json
from pathlib import Path

DATASET_DIR = '/content/drive/MyDrive/SignBridgeAI/dataset'
CSV_PATH = os.path.join(DATASET_DIR, 'iSign_v1.1.csv')
POSE_DIR = os.path.join(DATASET_DIR, 'poses')
VIDEO_DIR = os.path.join(DATASET_DIR, 'videos')
CACHE_DIR = os.path.join(DATASET_DIR, 'cache')
INDEX_PATH = os.path.join(CACHE_DIR, 'dataset_index.json')

os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
# Load CSV
print('Loading CSV...')
entries = []
with open(CSV_PATH, 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        uid = row['uid']
        text = row['text']
        base_uid = uid.split('_')[0] if '_' in uid else uid
        entries.append({'uid': uid, 'base_uid': base_uid, 'text': text})
print(f'  Loaded {len(entries)} entries')

In [ ]:
# Scan pose and video files
print('Scanning files...')
pose_files = {}
if os.path.exists(POSE_DIR):
    for f in Path(POSE_DIR).rglob('*.npy'):
        pose_files[f.stem] = str(f)

video_files = {}
if os.path.exists(VIDEO_DIR):
    for ext in ['*.mp4', '*.avi', '*.mov']:
        for f in Path(VIDEO_DIR).rglob(ext):
            video_files[f.stem] = str(f)

print(f'  Pose files: {len(pose_files)}')
print(f'  Video files: {len(video_files)}')

In [ ]:
# Build index
print('Building index...')
index = {
    'version': '1.0',
    'dataset': 'Exploration-Lab/iSign',
    'total_entries': len(entries),
    'found_poses': 0,
    'found_videos': 0,
    'entries': [],
}

for entry in entries:
    uid = entry['uid']
    base_uid = entry['base_uid']
    pose_path = pose_files.get(uid, pose_files.get(base_uid, ''))
    video_path = video_files.get(uid, video_files.get(base_uid, ''))
    
    idx_entry = {
        'uid': uid,
        'text': entry['text'],
        'pose_path': pose_path,
        'video_path': video_path,
    }
    index['entries'].append(idx_entry)
    if pose_path:
        index['found_poses'] += 1
    if video_path:
        index['found_videos'] += 1

# Save
with open(INDEX_PATH, 'w', encoding='utf-8') as f:
    json.dump(index, f, indent=2, ensure_ascii=False)

print(f'\nIndex saved: {INDEX_PATH}')
print(f'  Total entries: {index["total_entries"]}')
print(f'  Found poses: {index["found_poses"]}')
print(f'  Found videos: {index["found_videos"]}')

In [ ]:
# Verify index
print('Verifying index...')
with open(INDEX_PATH, 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print(f'  Entries: {len(loaded["entries"])}')
print(f'  Sample: {json.dumps(loaded["entries"][0], indent=2)[:200]}...')

# File size
size = os.path.getsize(INDEX_PATH)
print(f'  Index size: {size/1024:.1f} KB')
print('\nDataset preparation complete!')